# Phase 1: Data Preparation & Taxonomy Validation

**CSCI E-222 · Spring 2026**

This notebook covers every step in Phase 1:
1. Taxonomy inspection and label distribution
2. Amazon dataset download and category mapping
3. Preprocessing pipeline validation
4. Stratified train/val/test split
5. Class imbalance analysis and pos_weight computation

> **Flipkart holdout**: loaded and saved here but never inspected for label distribution until final evaluation.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
sns.set_theme(style='whitegrid', palette='muted')

print('Environment OK')

## 1. Taxonomy Inspection

In [ ]:
with open('../data/taxonomy.json') as f:
    taxonomy = json.load(f)

labels = taxonomy['labels']
print(f"Taxonomy version: {taxonomy['version']}")
print(f"Number of labels: {len(labels)}")
print()
for l in labels:
    print(f"  [{l['id']:2d}] {l['tag']:<20} {l['description']}")

In [ ]:
# Verify TaxonomyMapper loads cleanly
from data.taxonomy_mapper import TaxonomyMapper

mapper = TaxonomyMapper('../data/taxonomy.json')

# Spot-check a few mappings
test_cases = [
    ('Electronics', 'Bluetooth Speaker', 'Wireless speaker with deep bass'),
    ('Clothing_Shoes_and_Jewelry', 'Nike Air Max', 'Running shoes for men'),
    ('Clothing_Shoes_and_Jewelry', 'Gold Necklace', '18k gold pendant necklace'),
    ('Health_and_Household', 'Vitamin C 1000mg', 'Immune support supplement'),
    ('Home_and_Kitchen', 'Cast Iron Skillet', 'Pre-seasoned 12-inch pan'),
]

print(f"{'Category':<35} {'Title':<30} {'Tags'}")
print('-' * 90)
for cat, title, desc in test_cases:
    vec = mapper.map(cat, title, desc)
    tags = mapper.tag_names(vec)
    print(f"{cat:<35} {title:<30} {tags}")

## 2. Amazon Dataset Download

Downloads product metadata via HuggingFace Datasets. Raw files are cached to `data/raw/amazon/` and are gitignored.

In [ ]:
from data.amazon_loader import AmazonLoader

loader = AmazonLoader(mapper, cache_dir='../data/raw/amazon')

# Set max_products to control corpus size. ~80K is the target.
amazon_df = loader.load(max_products=80_000)

print(f"\nLoaded: {len(amazon_df):,} products")
amazon_df.head(3)

In [ ]:
# Category distribution
cat_counts = amazon_df['amazon_category'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
cat_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Product count')
ax.set_title('Amazon products per category')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/processed/category_distribution.png', dpi=150)
plt.show()

## 3. Preprocessing Pipeline Validation

In [ ]:
from transformers import AutoTokenizer
from agents.preprocessor import Preprocessor

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
preprocessor = Preprocessor(tokenizer, min_length=20, max_length=120)

# Run on a sample
sample = amazon_df.sample(5, random_state=42).to_dict('records')
for product in sample:
    result = preprocessor.run(product)
    if result:
        n_tokens = len(result['input_ids'])
        print(f"  [{n_tokens:3d} tokens] {result['raw_title'][:60]}")
    else:
        print(f"  [SKIPPED] {product.get('title', '')[:60]}")

In [ ]:
# Token length distribution across the full corpus
preprocessor.reset_dedup()
token_lengths = []
skipped = 0

for _, row in amazon_df.iterrows():
    result = preprocessor.run(row.to_dict())
    if result:
        token_lengths.append(len(result['input_ids']))
    else:
        skipped += 1

print(f"Processed: {len(token_lengths):,} | Skipped (dup/short): {skipped:,}")
print(f"Token length — mean: {np.mean(token_lengths):.1f} | median: {np.median(token_lengths):.0f} "
      f"| p5: {np.percentile(token_lengths, 5):.0f} | p95: {np.percentile(token_lengths, 95):.0f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(token_lengths, bins=50, color='steelblue', edgecolor='white')
ax.axvline(120, color='tomato', linestyle='--', label='Max (120)')
ax.axvline(20, color='orange', linestyle='--', label='Min (20)')
ax.set_xlabel('Token count')
ax.set_ylabel('Frequency')
ax.set_title('Token length distribution after preprocessing')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/token_length_dist.png', dpi=150)
plt.show()

## 4. Stratified Train / Val / Test Split

In [ ]:
from data.dataset_builder import DatasetBuilder

builder = DatasetBuilder(num_labels=30)

# Flipkart holdout: load your Flipkart CSV here before calling build().
# It must have 'title', 'description', and 'labels' columns.
# flipkart_df = pd.read_csv('../data/raw/flipkart/flipkart_products.csv')
flipkart_df = None  # swap in when the Flipkart data is available

splits = builder.build(amazon_df, flipkart_df)
train_df, val_df, test_df = splits['train'], splits['val'], splits['test']

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## 5. Class Imbalance Analysis

In [ ]:
with open('../data/processed/stats.json') as f:
    stats = json.load(f)

print(f"Average labels per product: {stats['avg_labels_per_product']}")
print(f"Label coverage (labels with ≥1 sample): {stats['label_coverage_pct']}%")
print()

tag_names = [l['tag'] for l in labels]
pos_counts = stats['per_label_pos_counts']
pos_weights = stats['per_label_pos_weights']

imbalance_df = pd.DataFrame({
    'label': tag_names,
    'pos_count': pos_counts,
    'pos_weight': pos_weights,
}).sort_values('pos_count', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Positive counts
axes[0].barh(imbalance_df['label'], imbalance_df['pos_count'], color='steelblue')
axes[0].set_xlabel('Positive samples (train set)')
axes[0].set_title('Positive count per label')

# Pos weights
axes[1].barh(imbalance_df['label'], imbalance_df['pos_weight'], color='coral')
axes[1].set_xlabel('pos_weight (neg/pos ratio)')
axes[1].set_title('BCEWithLogitsLoss pos_weight per label')

plt.tight_layout()
plt.savefig('../data/processed/class_imbalance.png', dpi=150)
plt.show()

print('\nTop 5 most imbalanced labels (highest pos_weight):')
print(imbalance_df.sort_values('pos_weight', ascending=False).head(5)[['label', 'pos_count', 'pos_weight']].to_string(index=False))

In [ ]:
# Verify no label leakage: each label should appear in all three splits
def label_coverage_per_split(df, name):
    matrix = np.array(df['labels'].tolist())
    covered = (matrix.sum(axis=0) > 0).sum()
    print(f"{name:8s}: {covered}/30 labels covered")

label_coverage_per_split(train_df, 'Train')
label_coverage_per_split(val_df,   'Val')
label_coverage_per_split(test_df,  'Test')
print()
print('Phase 1 complete. Splits saved to data/processed/')